# Stage 1: Filter Metadata

## Context: Trastuzumab Response Prediction in Breast Cancer

### Problem Statement
Trastuzumab is a targeted therapy for HER2-positive breast cancer, but not all patients respond to treatment. This project aims to predict treatment response using gene expression data from breast cancer patients, building a portfolio demonstrating bioinformatics data processing, quality control, and analysis skills.

### Data Source
We use datasets from the [Enlight study](https://github.com/PangeaResearch/enlight-data/tree/main), which compiled trastuzumab treatment response data across multiple breast cancer microarray experiments. Rather than manually downloading and normalizing raw data from NCBI GEO, we leverage **RefineBC (refine.bio)** - a uniformly processed collection of ~418K RNA samples (mostly microarray) aggregated from public repositories.

### Why RefineBC?
RefineBC provides pre-normalized, quality-controlled microarray data with consistent processing:

**Sample-level processing:**
- Background removal
- SCAN normalization (output is log2+1 transformed and centered: μ=0, σ=1)

**Experiment aggregation:**
- Filter genes expressed in <30% of samples and samples with <50% genes expressed
- Impute missing values using SVD
- Quantile normalization across experiments

This allows us to focus on analysis rather than time-consuming preprocessing. The RefineBC normalized compendia for *Homo sapiens* contains 418,073 samples × 14,549 genes and is available at: https://www.refine.bio/compendia/normalized

### Target Experiments
From the Enlight study supplementary data, we identified 4 trastuzumab breast cancer experiments from NCBI GEO:

| Experiment ID | Platform | Samples | Notes |
|:---:|:---:|:---:|:---|
| GSE66399 | Microarray | - | Parent study (contains GSE66305 with actual RNA data) |
| GSE37946 | Microarray | 50 | Breast cancer tissue samples |
| GSE42822 | Microarray | 91 | Breast cancer ultrasound (US) samples |
| GSE50948 | Microarray | 150 | Primary breast cancer samples |

**Note:** GSE66399 is a composite experiment. When viewing it on NCBI GEO (https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE66399), we find that the sub-experiment GSE66305 contains the actual RNA expression data we need.

### This Notebook: Stage 1
This notebook performs the first stage of our pipeline:

**Input:** `data/rb_obs.csv` (RefineBC metadata for all 418K samples)

**Process:**
1. Filter metadata to extract only samples from our 4 target experiments
2. Perform exploratory data analysis
3. Check for duplicate sample IDs (refinebio_accession_code = GSM IDs)
4. Extract row indexes for efficient data subsetting

**Output:**
- `data/refine_bio_obs_subset.csv` - Filtered metadata (~291 samples)
- `data/refine_bio_sample_indexes.npy` - NumPy array of row indexes

These indexes enable efficient extraction of the corresponding gene expression data from the full RefineBC dataset (stored separately as a Zarr database), avoiding the need to load the entire 15GB dataset into memory.

## Setup

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Set repository root
os.environ['BIO_REPPO_ROOT'] = r"C:\Users\Jesse\Google Drive\Learning\freelance\biodata_portfolio"
repo_root = Path(os.environ['BIO_REPPO_ROOT'])

print(f"Repository root: {repo_root}")

Repository root: C:\Users\Jesse\Google Drive\Learning\freelance\biodata_portfolio


## Load RefineBC Metadata

In [2]:
# Load the complete refinebio metadata
refine_bio_meta = pd.read_csv(repo_root / 'data/rb_obs.csv')

print(f"Total samples in RefineBC metadata: {len(refine_bio_meta):,}")
print(f"Columns: {refine_bio_meta.shape[1]}")
print(f"\nFirst few rows:")
refine_bio_meta.head()

C:\Users\Jesse\AppData\Local\Temp\ipykernel_24532\3009841235.py:2: DtypeWarning: Columns (0,1,7,18) have mixed types. Specify dtype option on import or set low_memory=False.
  refine_bio_meta = pd.read_csv(repo_root / 'data/rb_obs.csv')


Total samples in RefineBC metadata: 418,073
Columns: 25

First few rows:


,characteristic_BioSourceProvider,characteristic_BioSourceType,experiment_accession,is_microarray,refinebio_accession_code,refinebio_age,refinebio_cell_line,refinebio_compound,refinebio_disease,refinebio_disease_stage,...,refinebio_processor_version,refinebio_race,refinebio_sex,refinebio_source_archive_url,refinebio_source_database,refinebio_specimen_part,refinebio_subject,refinebio_time,refinebio_title,refinebio_treatment
0,The National Disease Research Interchange (NDRI),frozen_sample,E-AFMX-1,True,E-AFMX-1-BioSource:h1a,70.0,NaN,NaN,none,none,...,v1.4.7,NaN,male,https://www.ebi.ac.uk/arrayexpress/json/v3/exp...,ARRAY_EXPRESS,prefrontal cortex,NaN,NaN,Extract:h1a,NaN
1,The National Disease Research Interchange (NDRI),frozen_sample,E-AFMX-1,True,E-AFMX-1-BioSource:h2a,45.0,NaN,NaN,none,none,...,v1.4.7,NaN,male,https://www.ebi.ac.uk/arrayexpress/json/v3/exp...,ARRAY_EXPRESS,prefrontal cortex,NaN,NaN,Extract:h2a,NaN
2,The National Disease Research Interchange (NDRI),frozen_sample,E-AFMX-1,True,E-AFMX-1-BioSource:h3a,45.0,NaN,NaN,none,none,...,v1.4.7,NaN,male,https://www.ebi.ac.uk/arrayexpress/json/v3/exp...,ARRAY_EXPRESS,prefrontal cortex,NaN,NaN,Extract:h3a,NaN
3,The National Disease Research Interchange (NDRI),frozen_sample,E-AFMX-1,True,E-AFMX-1-BioSource:h4a,63.0,NaN,NaN,none,none,...,v1.4.7,NaN,male,https://www.ebi.ac.uk/arrayexpress/json/v3/exp...,ARRAY_EXPRESS,prefrontal cortex,NaN,NaN,Extract:h4a,NaN
4,The National Disease Research Interchange (NDRI),frozen_sample,E-AFMX-1,True,E-AFMX-1-BioSource:h5a,65.0,NaN,NaN,none,none,...,v1.4.7,NaN,male,https://www.ebi.ac.uk/arrayexpress/json/v3/exp...,ARRAY_EXPRESS,prefrontal cortex,NaN,NaN,Extract:h5a,NaN


## Exploratory Data Analysis

In [3]:
# Check data types and non-null counts
print("Dataset Info:")
refine_bio_meta.info()

print("\n" + "="*50)
print("Key Columns:")
print("="*50)
print(f"Unique experiments: {refine_bio_meta['experiment_accession'].nunique()}")
print(f"Unique samples (GSM IDs): {refine_bio_meta['refinebio_accession_code'].nunique()}")
print(f"\nSample of experiment IDs:")
print(refine_bio_meta['experiment_accession'].value_counts().head(10))

Dataset Info:


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418073 entries, 0 to 418072
Data columns (total 25 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   characteristic_BioSourceProvider  200 non-null     object 
 1   characteristic_BioSourceType      993 non-null     object 
 2   experiment_accession              418073 non-null  object 
 3   is_microarray                     418073 non-null  bool   
 4   refinebio_accession_code          418073 non-null  object 
 5   refinebio_age                     68318 non-null   float64
 6   refinebio_cell_line               58406 non-null   object 
 7   refinebio_compound                2812 non-null    object 
 8   refinebio_disease                 73065 non-null   object 
 9   refinebio_disease_stage           46402 non-null   object 
 10  refinebio_organism                418073 non-null  object 
 11  refinebio_platform                418073 non-null  o

## Filter for Target Experiments

In [3]:
# Define target experiments
series_list = ["GSE66399", "GSE37946", "GSE42822", "GSE50948",'GSE66305']

# Filter metadata
mask = refine_bio_meta['experiment_accession'].str.contains("|".join(series_list), na=False)
subset_refine_bio_meta = refine_bio_meta[mask]

print(f"Samples matching target experiments: {len(subset_refine_bio_meta)}")
print(f"\nBreakdown by experiment:")
print(subset_refine_bio_meta['experiment_accession'].value_counts())
print(f"\nFiltered data shape: {subset_refine_bio_meta.shape}")

subset_refine_bio_meta.head()

Samples matching target experiments: 374

Breakdown by experiment:
experiment_accession
GSE50948    150
GSE42822     91
GSE66305     83
GSE37946     50
Name: count, dtype: int64

Filtered data shape: (374, 25)


,characteristic_BioSourceProvider,characteristic_BioSourceType,experiment_accession,is_microarray,refinebio_accession_code,refinebio_age,refinebio_cell_line,refinebio_compound,refinebio_disease,refinebio_disease_stage,...,refinebio_processor_version,refinebio_race,refinebio_sex,refinebio_source_archive_url,refinebio_source_database,refinebio_specimen_part,refinebio_subject,refinebio_time,refinebio_title,refinebio_treatment
10650,NaN,NaN,GSE42822,True,GSM1050577,32.197,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US001,NaN
10651,NaN,NaN,GSE42822,True,GSM1050578,42.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US005,NaN
10652,NaN,NaN,GSE42822,True,GSM1050579,36.739,NaN,NaN,NaN,NaN,...,v1.23.4-hotfix,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US011,NaN
10653,NaN,NaN,GSE42822,True,GSM1050580,47.099,NaN,NaN,NaN,NaN,...,v1.4.9,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US014,NaN
10654,NaN,NaN,GSE42822,True,GSM1050581,44.000,NaN,NaN,NaN,NaN,...,v1.4.8,NaN,NaN,NaN,GEO,NaN,NaN,NaN,breast cancer US016,NaN


## Inspect Filtered Metadata

In [4]:
# Check which columns have data in the filtered subset
print("Non-null counts in filtered subset:")
subset_refine_bio_meta.info()

print("\n" + "="*50)
print("Sample titles:")
print("="*50)
print(subset_refine_bio_meta['refinebio_title'].head(10))

Non-null counts in filtered subset:
<class 'pandas.core.frame.DataFrame'>
Index: 374 entries, 10650 to 304143
Data columns (total 25 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   characteristic_BioSourceProvider  0 non-null      object 
 1   characteristic_BioSourceType      0 non-null      object 
 2   experiment_accession              374 non-null    object 
 3   is_microarray                     374 non-null    bool   
 4   refinebio_accession_code          374 non-null    object 
 5   refinebio_age                     291 non-null    float64
 6   refinebio_cell_line               0 non-null      object 
 7   refinebio_compound                0 non-null      object 
 8   refinebio_disease                 83 non-null     object 
 9   refinebio_disease_stage           83 non-null     object 
 10  refinebio_organism                374 non-null    object 
 11  refinebio_platform               

## Check for Duplicates

In [5]:
# Check for duplicate GSM sample IDs
has_duplicates = subset_refine_bio_meta['refinebio_accession_code'].duplicated().any()

print(f"Contains duplicate refinebio_accession_code (GSM IDs): {has_duplicates}")

if has_duplicates:
    duplicates = subset_refine_bio_meta[subset_refine_bio_meta['refinebio_accession_code'].duplicated(keep=False)]
    print(f"\nNumber of duplicate samples: {len(duplicates)}")
    print("\nDuplicate samples:")
    print(duplicates[['refinebio_accession_code', 'experiment_accession', 'refinebio_title']])
else:
    print("✓ No duplicates found - each sample is unique")

Contains duplicate refinebio_accession_code (GSM IDs): False
✓ No duplicates found - each sample is unique


## Extract Row Indexes

In [7]:
# Get row indexes from the original dataframe
indexes = np.array(subset_refine_bio_meta.index.values).astype(int)

print(f"Number of indexes extracted: {len(indexes)}")
print(f"Index range: {indexes.min()} to {indexes.max()}")
print(f"\nFirst 10 indexes:")
print(indexes[:10])
print(f"\nLast 10 indexes:")
print(indexes[-10:])

Number of indexes extracted: 374
Index range: 10650 to 304143

First 10 indexes:
[10650 10651 10652 10653 10654 10655 10656 10657 10658 10659]

Last 10 indexes:
[304134 304135 304136 304137 304138 304139 304140 304141 304142 304143]


## Save Outputs

In [8]:
# Save filtered metadata to CSV
output_path = repo_root / 'data/refine_bio_obs_subset.csv'
subset_refine_bio_meta.to_csv(output_path, index=False)
print(f"✓ Saved filtered metadata to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

# Save indexes to numpy file for easy loading
indexes_path = repo_root / 'data/refine_bio_sample_indexes.npy'
np.save(indexes_path, indexes)
print(f"\n✓ Saved sample indexes to: {indexes_path}")
print(f"  Array shape: {indexes.shape}")

✓ Saved filtered metadata to: C:\Users\Jesse\Google Drive\Learning\freelance\biodata_portfolio\data\refine_bio_obs_subset.csv
  File size: 84.7 KB

✓ Saved sample indexes to: C:\Users\Jesse\Google Drive\Learning\freelance\biodata_portfolio\data\refine_bio_sample_indexes.npy
  Array shape: (374,)


## Summary

In [9]:
print("="*70)
print("STAGE 1 COMPLETE: Filter Metadata")
print("="*70)
print(f"✓ Filtered {len(refine_bio_meta):,} samples down to {len(subset_refine_bio_meta)}")
print(f"✓ Target experiments: {', '.join(series_list)}")
print(f"✓ No duplicate GSM IDs found")
print(f"✓ Extracted {len(indexes)} row indexes")
print(f"✓ Saved filtered metadata: data/refine_bio_obs_subset.csv")
print(f"✓ Saved indexes array: data/refine_bio_sample_indexes.npy")
print("\nReady for Stage 2: QC Gene Expression Data")

STAGE 1 COMPLETE: Filter Metadata
✓ Filtered 418,073 samples down to 374
✓ Target experiments: GSE66399, GSE37946, GSE42822, GSE50948, GSE66305
✓ No duplicate GSM IDs found
✓ Extracted 374 row indexes
✓ Saved filtered metadata: data/refine_bio_obs_subset.csv
✓ Saved indexes array: data/refine_bio_sample_indexes.npy

Ready for Stage 2: QC Gene Expression Data
